# ДЗ 13 — Работа с API VirusTotal (AV/FW)


## 1) Подготовка окружения


In [ ]:
import os
import json
import time
from datetime import datetime
from pathlib import Path

import requests
from requests import Response
from getpass import getpass

VT_BASE_URL = "https://www.virustotal.com/api/v3"

def get_api_key() -> str:
    """Получить API-ключ из окружения или запросить у пользователя."""
    key = os.getenv("VT_API_KEY", "").strip()
    if key:
        return key
    return getpass("Введите VT API key (ввод скрыт): ").strip()

def vt_headers(api_key: str) -> dict:
    return {"x-apikey": api_key}

def vt_request(method: str, endpoint: str, api_key: str, **kwargs) -> dict:
    """Единая обёртка над запросами к VT API с обработкой ошибок."""
    url = endpoint if endpoint.startswith("http") else f"{VT_BASE_URL}{endpoint}"
    resp: Response = requests.request(method, url, headers=vt_headers(api_key), timeout=60, **kwargs)
    try:
        data = resp.json()
    except Exception:
        data = {"raw_text": resp.text}

    if resp.status_code >= 400:
        raise RuntimeError(
            f"VirusTotal API error: HTTP {resp.status_code}\n"
            f"URL: {url}\n"
            f"Response: {json.dumps(data, ensure_ascii=False, indent=2)[:2000]}"
        )
    return data

def pretty_print(obj: dict, max_chars: int = 20000) -> None:
    s = json.dumps(obj, ensure_ascii=False, indent=2)
    if len(s) > max_chars:
        print(s[:max_chars] + "\n... (truncated) ...")
    else:
        print(s)

def save_json(obj: dict, out_dir: str = "outputs", prefix: str = "vt") -> str:
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    path = Path(out_dir) / f"{prefix}_{ts}.json"
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")
    return str(path)


## 3) Запрос: отчёт по файлу (по хэшу)

Ниже пример для тестовой строки EICAR (безвредный тестовый файл антивирусов):  
- **SHA-256:** `275a021bbfb6489e54d471899f7db9d1663fc695ec2fe2a2c4538aabf651fd0f`



In [3]:
api_key = get_api_key()
print("API key loaded:", "YES" if api_key else "NO")

API key loaded: YES


In [ ]:
def get_file_report(file_id_or_hash: str, api_key: str) -> dict:
    """Получить отчёт VT по файлу (по hash или VT file id)."""
    return vt_request("GET", f"/files/{file_id_or_hash}", api_key)

FILE_ID = "275a021bbfb6489e54d471899f7db9d1663fc695ec2fe2a2c4538aabf651fd0f"
report = get_file_report(FILE_ID, api_key)

print("Получен JSON-ответ (files/{id}). Превью:")
pretty_print(report, max_chars=8000)

saved_path = save_json(report, prefix="vt_file_report")
print("JSON сохранён в:", saved_path)


Получен JSON-ответ (files/{id}). Превью:
{
  "data": {
    "id": "275a021bbfb6489e54d471899f7db9d1663fc695ec2fe2a2c4538aabf651fd0f",
    "type": "file",
    "links": {
      "self": "https://www.virustotal.com/api/v3/files/275a021bbfb6489e54d471899f7db9d1663fc695ec2fe2a2c4538aabf651fd0f"
    },
    "attributes": {
      "magika": "POWERSHELL",
      "size": 68,
      "last_analysis_stats": {
        "malicious": 67,
        "suspicious": 0,
        "undetected": 2,
        "harmless": 0,
        "timeout": 0,
        "confirmed-timeout": 0,
        "failure": 0,
        "type-unsupported": 7
      },
      "type_description": "Powershell",
      "last_modification_date": 1772648678,
      "sigma_analysis_summary": {
        "Sigma Integrated Rule Set (GitHub)": {
          "high": 0,
          "medium": 0,
          "critical": 0,
          "low": 1
        }
      },
      "reputation": 3725,
      "known_distributors": {
        "distributors": [
          "Offensive Security"
      

### 3.1) Короткое резюме по детектам

Вытащим сводку по `last_analysis_stats` и (если есть) первые N движков с их вердиктами.


In [5]:
def extract_detection_summary(file_report: dict, top_n: int = 10) -> dict:
    attrs = (file_report.get("data") or {}).get("attributes") or {}
    stats = attrs.get("last_analysis_stats") or {}
    results = attrs.get("last_analysis_results") or {}

    # Сортируем движки: сначала malicious/suspicious, потом остальные
    priority = {"malicious": 0, "suspicious": 1, "undetected": 2, "harmless": 3, "timeout": 4, "type-unsupported": 5}
    items = []
    for engine, r in results.items():
        cat = r.get("category", "unknown")
        items.append({
            "engine": engine,
            "category": cat,
            "result": r.get("result"),
            "method": r.get("method"),
        })
    items.sort(key=lambda x: (priority.get(x["category"], 99), x["engine"].lower()))
    return {
        "stats": stats,
        "top_engines": items[:top_n],
        "permalink": attrs.get("permalink") or ((file_report.get("data") or {}).get("links") or {}).get("self"),
    }

summary = extract_detection_summary(report, top_n=12)
pretty_print(summary, max_chars=20000)

saved_summary_path = save_json(summary, prefix="vt_summary")
print("Summary JSON сохранён в:", saved_summary_path)


{
  "stats": {
    "malicious": 67,
    "suspicious": 0,
    "undetected": 2,
    "harmless": 0,
    "timeout": 0,
    "confirmed-timeout": 0,
    "failure": 0,
    "type-unsupported": 7
  },
  "top_engines": [
    {
      "engine": "AhnLab-V3",
      "category": "malicious",
      "result": "Virus/EICAR_Test_File",
      "method": "blacklist"
    },
    {
      "engine": "Alibaba",
      "category": "malicious",
      "result": "Virus:Win32/EICAR.A",
      "method": "blacklist"
    },
    {
      "engine": "alibabacloud",
      "category": "malicious",
      "result": "Engtest:Multi/Eicar",
      "method": "blacklist"
    },
    {
      "engine": "ALYac",
      "category": "malicious",
      "result": "Misc.Eicar-Test-File",
      "method": "blacklist"
    },
    {
      "engine": "Antiy-AVL",
      "category": "malicious",
      "result": "TestFile/Win32.EICAR",
      "method": "blacklist"
    },
    {
      "engine": "APEX",
      "category": "malicious",
      "result": "EICAR Anti